In [1]:
import yfinance as yf
import pandas as pd

stocks = [
"ADANIENT.NS","ADANIPORTS.NS","APOLLOHOSP.NS","ASIANPAINT.NS","AXISBANK.NS",
"BAJAJ-AUTO.NS","BAJAJFINSV.NS","BAJFINANCE.NS","BEL.NS","BHARTIARTL.NS",
"CIPLA.NS","COALINDIA.NS","DRREDDY.NS","EICHERMOT.NS","GRASIM.NS",
"HCLTECH.NS","HDFCBANK.NS","HDFCLIFE.NS","HINDALCO.NS","HINDUNILVR.NS",
"ICICIBANK.NS","INDIGO.NS","INFY.NS","ITC.NS","JIOFIN.NS","JSWSTEEL.NS",
"KOTAKBANK.NS","LT.NS","M&M.NS","MARUTI.NS","MAXHEALTH.NS","NESTLEIND.NS",
"NTPC.NS","ONGC.NS","POWERGRID.NS","RELIANCE.NS","SBILIFE.NS","SBIN.NS",
"SHRIRAMFIN.NS","SUNPHARMA.NS","TATACONSUM.NS","TATASTEEL.NS","TCS.NS",
"TECHM.NS","TITAN.NS","TMPV.NS","TRENT.NS","ULTRACEMCO.NS","WIPRO.NS","ETERNAL.NS"
]

all_data = []

for stock in stocks:

    df = yf.download(
        stock,
        start="2020-01-01",
        end="2025-12-31",
        progress=False
    )

    if df.empty:
        continue

    df.reset_index(inplace=True)

    # remove multiindex columns
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    df["Stock"] = stock

    
    # FIX missing Adj Close
    
    if "Adj Close" not in df.columns:
        df["Adj Close"] = df["Close"]

    df = df[["Date","Stock","Open","High","Low","Close","Adj Close","Volume"]]

    all_data.append(df)

final_df = pd.concat(all_data, ignore_index=True)

#final_df.to_excel("Indian_Nifty50_Stock_Data1.xlsx", index=False)

print("Excel file saved successfully")

Excel file saved successfully


In [2]:
final_df.to_excel("Indian_Nifty50_Stock_Data.xlsx", index=False)

print("Excel file saved successfully")

Excel file saved successfully


In [3]:
final_df

,Date,Stock,Open,High,Low,Close,Adj Close,Volume
0,2020-01-01,ADANIENT.NS,207.025666,208.461965,204.697859,205.886536,205.886536,1553127
1,2020-01-02,ADANIENT.NS,206.035082,211.185956,205.539805,209.204849,209.204849,2991937
2,2020-01-03,ADANIENT.NS,208.263851,210.344020,203.855892,206.332275,206.332275,2512421
3,2020-01-06,ADANIENT.NS,205.787467,205.787467,195.881933,197.664932,197.664932,4353179
4,2020-01-07,ADANIENT.NS,198.655454,203.756798,198.655454,202.122391,202.122391,2966120
...,...,...,...,...,...,...,...,...
71386,2025-12-23,ETERNAL.NS,287.049988,288.000000,284.000000,284.350006,284.350006,12762620
71387,2025-12-24,ETERNAL.NS,283.500000,286.000000,283.100006,284.850006,284.850006,12973810
71388,2025-12-26,ETERNAL.NS,281.799988,284.649994,279.700012,281.750000,281.750000,15744490
71389,2025-12-29,ETERNAL.NS,281.500000,286.049988,281.500000,282.850006,282.850006,13175250


In [4]:
nifty = yf.download("^NSEI", start="2020-01-01", end="2025-12-31")

nifty.reset_index(inplace=True)

# Flatten 
if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = nifty.columns.get_level_values(0)

nifty = nifty[["Date", "Close"]]
nifty.rename(columns={"Close": "Index_Close"}, inplace=True)

# Save to Excel
nifty.to_excel("nifty50.xlsx", index=False)

[*********************100%***********************]  1 of 1 completed


In [5]:
fundamental_data = []

for stock in stocks:
    ticker = yf.Ticker(stock)
    info = ticker.info
    
    fundamental_data.append({
        "Stock": stock,
        "Market_Cap": info.get("marketCap"),
        "PE_Ratio": info.get("trailingPE"),
        "PB_Ratio": info.get("priceToBook"),
        "ROE": info.get("returnOnEquity"),
        "ROA": info.get("returnOnAssets"),
        "Debt_to_Equity": info.get("debtToEquity"),
        "EPS": info.get("trailingEps"),
        "Revenue": info.get("totalRevenue"),
        "Profit_Margin": info.get("profitMargins"),
        "Dividend_Yield": info.get("dividendYield")
    })

fundamental_df = pd.DataFrame(fundamental_data)

print(fundamental_df.head())

           Stock     Market_Cap   PE_Ratio   PB_Ratio      ROE      ROA  \
0    ADANIENT.NS  2569423290368  17.040918   4.233747      NaN      NaN   
1  ADANIPORTS.NS  3170017214464  23.858160   4.431754  0.19407      NaN   
2  APOLLOHOSP.NS  1089959559168  60.508460  11.985870      NaN      NaN   
3  ASIANPAINT.NS  2176200736768  56.571140  11.109915      NaN      NaN   
4    AXISBANK.NS  3801037930496  14.514252   1.833047  0.13625  0.01563   

   Debt_to_Equity     EPS       Revenue  Profit_Margin  Dividend_Yield  
0         177.767  110.71  949951594496        0.14111            0.07  
1          81.556   57.67  364866306048        0.34236            0.52  
2          83.606  125.28  242151997440        0.07441            0.27  
3          17.609   40.13  346495287296        0.11098            1.13  
4             NaN   84.20  768435683328        0.34170            0.08  


In [6]:
fundamental_df.fillna(0, inplace=True)

In [45]:
fundamental_df.to_excel("nifty50_fundamental_data.xlsx", index=False)

In [7]:
import yfinance as yf
import pandas as pd

data = []

for stock in stocks:
    print(f"Fetching {stock}...")
    
    ticker = yf.Ticker(stock)
    
    try:
        info = ticker.info
        
        sector = info.get("sector")
        industry = info.get("industry")
        
        # Handle missing values
        if not sector:
            sector = "Unknown"
        if not industry:
            industry = "Unknown"
    
    except:
        sector = "Unknown"
        industry = "Unknown"
    
    data.append({
        "Stock": stock,
        "Sector": sector,
        "Industry": industry
    })

sector_industry_df = pd.DataFrame(data)

print(sector_industry_df)

Fetching ADANIENT.NS...
Fetching ADANIPORTS.NS...
Fetching APOLLOHOSP.NS...
Fetching ASIANPAINT.NS...
Fetching AXISBANK.NS...
Fetching BAJAJ-AUTO.NS...
Fetching BAJAJFINSV.NS...
Fetching BAJFINANCE.NS...
Fetching BEL.NS...
Fetching BHARTIARTL.NS...
Fetching CIPLA.NS...
Fetching COALINDIA.NS...
Fetching DRREDDY.NS...
Fetching EICHERMOT.NS...
Fetching GRASIM.NS...
Fetching HCLTECH.NS...
Fetching HDFCBANK.NS...
Fetching HDFCLIFE.NS...
Fetching HINDALCO.NS...
Fetching HINDUNILVR.NS...
Fetching ICICIBANK.NS...
Fetching INDIGO.NS...
Fetching INFY.NS...
Fetching ITC.NS...
Fetching JIOFIN.NS...
Fetching JSWSTEEL.NS...
Fetching KOTAKBANK.NS...
Fetching LT.NS...
Fetching M&M.NS...
Fetching MARUTI.NS...
Fetching MAXHEALTH.NS...
Fetching NESTLEIND.NS...
Fetching NTPC.NS...
Fetching ONGC.NS...
Fetching POWERGRID.NS...
Fetching RELIANCE.NS...
Fetching SBILIFE.NS...
Fetching SBIN.NS...
Fetching SHRIRAMFIN.NS...
Fetching SUNPHARMA.NS...
Fetching TATACONSUM.NS...
Fetching TATASTEEL.NS...
Fetching TCS.N

In [8]:
fundamental_df = fundamental_df.merge(
    sector_industry_df,
    on="Stock",
    how="left"
)

In [9]:
print(fundamental_df.head())

           Stock     Market_Cap   PE_Ratio   PB_Ratio      ROE      ROA  \
0    ADANIENT.NS  2569423290368  17.040918   4.233747  0.00000  0.00000   
1  ADANIPORTS.NS  3170017214464  23.858160   4.431754  0.19407  0.00000   
2  APOLLOHOSP.NS  1089959559168  60.508460  11.985870  0.00000  0.00000   
3  ASIANPAINT.NS  2176200736768  56.571140  11.109915  0.00000  0.00000   
4    AXISBANK.NS  3801037930496  14.514252   1.833047  0.13625  0.01563   

   Debt_to_Equity     EPS       Revenue  Profit_Margin  Dividend_Yield  \
0         177.767  110.71  949951594496        0.14111            0.07   
1          81.556   57.67  364866306048        0.34236            0.52   
2          83.606  125.28  242151997440        0.07441            0.27   
3          17.609   40.13  346495287296        0.11098            1.13   
4           0.000   84.20  768435683328        0.34170            0.08   

               Sector                 Industry  
0              Energy             Thermal Coal  
1     

In [10]:
fundamental_df.to_excel("nifty50_fundamental_data.xlsx", index=False)